In [2]:
# For this step we will use pyscenic on the data in order to execute the tutorial

<https://pyscenic.readthedocs.io/en/latest/tutorial.html>

<https://github.com/aertslab/SCENICprotocol/blob/master/notebooks/SCENIC%20Protocol%20-%20Case%20study%20-%20Mouse%20brain%20data%20set.ipynb>

But I'll connect to jing's computer and see the steps that I did last time I ran SCENIC

In [ ]:
# Add here all the following analysis.

# do this in bash:
# adjacencies
TOTCD8
sudo docker run -it --rm \
    -v /home/bisquared/scenic/totcd8:/home/bisquared/scenic/totcd8 \
    aertslab/pyscenic:0.12.1 pyscenic grn \
        --num_workers 25 \
        -o /home/bisquared/scenic/totcd8/expr_mat.adjacencies.tsv \
        /home/bisquared/scenic/totcd8/expression_matrix.tsv \
        /home/bisquared/scenic/totcd8/TF_names_v_1.01.txt

# regulons
TOTCD8
sudo docker run -it --rm \
    -v /home/bisquared/scenic/totcd8:/home/bisquared/scenic/totcd8 \
    aertslab/pyscenic:0.12.1 pyscenic ctx \
        /home/bisquared/scenic/totcd8/expr_mat.adjacencies.tsv \
        /home/bisquared/scenic/totcd8/hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather \
        /home/bisquared/scenic/totcd8/hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather \
        --annotations_fname /home/bisquared/scenic/totcd8/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl \
        --expression_mtx_fname /home/bisquared/scenic/totcd8/expression_matrix.tsv \
        --mode "custom_multiprocessing" \
        --output /home/bisquared/scenic/totcd8/regulons.csv \
        --num_workers 25


AUC
TOTCD8
sudo docker run -it --rm \
    -v /home/bisquared/scenic/totcd8:/home/bisquared/scenic/totcd8 \
    aertslab/pyscenic:0.12.1 pyscenic aucell \
        /home/bisquared/scenic/totcd8/filtered.loom \
        /home/bisquared/scenic/totcd8/regulons.csv \
        -o /home/bisquared/scenic/totcd8/final.loom \
        --num_workers 5


So we end up having the regulons.csv as well as the expression adjacency matrix which contains our obtained regulons.
Let's load the data in here and provide the regulon analysis on inferenced GRNs from our data.

In [14]:
import pandas as pd
import numpy as np

In [15]:
auc_mtx = pd.read_csv("/home/roger/send_data/scenic/cd4/final/auc_mtx.csv", index_col=0)
adj_mtx = pd.read_csv("/home/roger/send_data/scenic/cd4/final/expr_mat.adjacencies.tsv", sep="\t")
regulons = pd.read_csv("/home/roger/send_data/scenic/cd4/final/regulons.csv")

In [20]:
# Scenic pruning step

# Read
grn = pd.read_csv("/home/roger/send_data/scenic/cd4/final/expr_mat.adjacencies.tsv", index_col=False, sep='\t').rename(columns={'TF': 'source', 'importance': 'score'})
#grn = grn[grn["score"] > 0.001]
top_n = int(np.ceil(grn.shape[0] * 0.1))
grn = grn.sort_values('score', ascending=False).head(top_n)

cols = ['source', 'target', 'score']

# Filter by enriched TFs
reg = pd.read_csv("/home/roger/send_data/scenic/cd4/final/regulons.csv")
reg = reg.iloc[2:, [0, 8]]
reg.columns = ['source', 'target']
reg['target'] = reg['target'].str.split(',')
reg_exp = reg.explode('target')
reg_exp['target'] = reg_exp['target'].str.replace(r"[\[\(\)' ]", "", regex=True)
grn = pd.merge(grn, reg_exp, on=['source', 'target'], how='inner')

# Format
grn = grn[cols]
grn = grn.sort_values(['source', 'target'])


In [22]:
import decoupler as dc
dc.mt.ulm(data=adata, net=grn)
score = dc.pp.get_obsm(adata=adata, key="score_ulm")

/home/roger/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'adata' is not defined

In [ ]:
tfs = dc.tl.rankby_order(adata=score, order="dpt_pseudotime", stat="dcor",)
tfs

In [ ]:
top_tfs = tfs.head(10)["name"].to_list()
sc.pl.umap(score, color=top_tfs, cmap="coolwarm", ncols=5, title=[f"{t} score" for t in top_tfs]) #vmin=-5, vmax=5, 

In [ ]:
bin_tfs = dc.pp.bin_order( adata=score, order="dpt_pseudotime", names=top_tfs, label="celltype", )
dc.pl.order( df=bin_tfs, mode="line", figsize=(6, 3), )

In [ ]:
dc.pl.network(
    net=collectri,
    data=data,
    score=tf_acts,
    sources=["SRF", "MYOCD", "IRF1", "SKIL"],
    targets=10,
    figsize=(5, 5),
    vcenter=True,
    by_abs=True,
    size_node=10,
)